## Setup

In [ ]:
import sqlite3
# Create a connection to the database
db_connection = sqlite3.connect('../experiment.db')

# set info level logging
from logging import basicConfig, INFO, getLogger
basicConfig(level=INFO)
logger = getLogger(__name__)

import sys
import os

# Assuming the notebook is in the same directory as the 'custom' package
sys.path.append(os.path.abspath('..'))

## Plot results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

test_name = "wiqiqa_ragtag_gpt35turbo"

def get_data(conn):
    query = f"""
    SELECT 
        re.test_run_id,
        tr.description as test_run_name,
        re.question_id,
        q.question as question_content,
        re.response_id,
        r.response as response_content,
        re.test_eval_config_id,
        re.eval_score,
        ef.name as eval_function_name
    FROM response_evals re
    JOIN test_runs tr ON re.test_run_id = tr.id
    JOIN questions q ON re.question_id = q.id
    JOIN responses r ON re.response_id = r.id
    JOIN test_eval_configs tec ON re.test_eval_config_id = tec.id
    JOIN eval_functions ef ON tec.eval_function_id = ef.id
    WHERE tr.description = '{test_name}'
    """
    return pd.read_sql_query(query, conn)


def analyze_data(df):
    # Set the style for all plots
    plt.style.use('ggplot')

    # Heatmap of scores by test function
    # Create score bins
    decimal_precision = 2
    score_bins = np.round(np.linspace(0, 1, 11), decimal_precision)
    bin_labels = [f"{score:.{decimal_precision}f}" for score in score_bins[:-1]]

    df['score_bin'] = pd.cut(df['eval_score'], bins=score_bins, labels=bin_labels, include_lowest=True)

    # Create a pivot table
    pivot_data = df.pivot_table(
        values='eval_score', 
        index='score_bin', 
        columns='eval_function_name', 
        aggfunc='count',
        fill_value=0
    ).sort_index(ascending=False)

    # Create the heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_data, cmap="Blues", annot=True, fmt='.2f', cbar_kws={'label': 'Normalized Count'})

    plt.title(f"Evaluation Scores for {test_name}")
    plt.xlabel("Test Function Name")
    plt.ylabel("Score Range")
    plt.tight_layout()
    plt.show()

    # 5. Summary statistics
    summary_stats = df.groupby('eval_function_name')['eval_score'].describe()
    summary_stats.to_csv('summary_statistics.csv')
    print("Summary Statistics:")
    print(summary_stats)

df = get_data(db_connection)
analyze_data(df)